# 🔍 FileMaker Layout & Record Diagnostics
This notebook connects to a FileMaker database, lists layouts, checks for available fields, and safely attempts record access without crashing.

In [1]:
import requests
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


In [2]:
HOST = "10.173.38.223"
DATABASE = "Laser Shock"
USERNAME = "your_JHED_username"
PASSWORD = "your_JHED_password"
BASE_URL = f"https://{HOST}/fmi/data/vLatest/databases/{DATABASE}"


In [3]:
login_url = f"{BASE_URL}/sessions"
headers = {'Content-Type': 'application/json'}

response = requests.post(login_url, headers=headers, auth=(USERNAME, PASSWORD), verify=False)
response.raise_for_status()

token = response.json()["response"]["token"]
print("Access token retrieved successfully.")


Access token retrieved successfully.


In [4]:
layout_url = f"{BASE_URL}/layouts"
layout_headers = {
    "Authorization": f"Bearer {token}"
}

layout_response = requests.get(layout_url, headers=layout_headers, verify=False)
layout_response.raise_for_status()

layouts = layout_response.json()["response"]["layouts"]
print("Available layouts:")
for layout in layouts:
    print("-", layout["name"])


Available layouts:
- Laser shock
- Launch Package
- Inventory
- Spacer Cutting Program
- Glass List
- Sample
- Flyer Stack
- Laser Characteristics
- Experiment
- Glass ID
- Epoxy ID
- Foil ID
- Spacer ID
- Experimenter
- Flyer Cutting Program
- Compression Methods
- Femtosecond Laser
- Ballistic Stack Fabrication
- Ballistic Launch Package


In [5]:
from fmrest import Server

def test_layout(layout_name):
    print(f"\n🔍 Testing layout: {layout_name}")
    try:
        server = Server(
            url=f"https://{HOST}",
            user=USERNAME,
            password=PASSWORD,
            database=DATABASE,
            layout=layout_name,
            verify_ssl=False,
            api_version='vLatest'
        )
        server._token = token
        server._logged_in = True

        print("  ➤ Fetching layout fields...")
        layout_info = server.get_layout(layout_name)
        if not layout_info:
            print("  ⚠ No fields found on layout.")
        else:
            for field in layout_info:
                print(f"  - {field['name']} (Type: {field['result']})")

        print("  ➤ Fetching a few records safely...")
        records = server.get_records()
        for i, record in enumerate(records):
            print(f"    Record {i + 1}:")
            for k in record.keys():
                print(f"      {k}: {record[k]}")
            if i >= 2:
                break

    except Exception as e:
        print(f"  ❌ Error with layout '{layout_name}':", e)

# Test all layouts
for layout in layouts:
    test_layout(layout["name"])



🔍 Testing layout: Laser shock
  ➤ Fetching layout fields...
  ⚠ No fields found on layout.
  ➤ Fetching a few records safely...
    Record 1:
      recordId: 82
      modId: 0

🔍 Testing layout: Launch Package
  ➤ Fetching layout fields...
  - Performed By (Type: text)
  - Date (Type: date)
  - Launch ID (Type: text)
  - Spacer Flag (Type: text)
  - Sample Flag (Type: text)
  - Flyer ID (Type: text)
  - Sample Name (Type: text)
  - Spacer Type (Type: text)
  - Flyer Column Location (Type: text)
  - Spacer Attachment Method (Type: text)
  - Sample Attachment Method (Type: text)
  - Flyer Row Location (Type: text)
  - Spacer Attachment Adhesive (Type: text)
  - Cutting Procedures Used (Type: text)
  - Sample Diameter (Type: number)
  - Spacer Cutting Program (Type: text)
  - Sample Impact Orientation (Type: text)
  - Spacer Inner Diameter (Type: number)
  - Sample Location (Type: text)
  - Spacer Outer Diameter (Type: number)
  - Sample Location based Order (Type: text)
  - Spacer Cutti